# Exploratory Analysis — TfL NUMBAT (notebook)

Reproduces the key EDA steps from the CLI/scripts. **Requires**
`data/processed/cleaned_data.parquet` (run `python scripts/profile_dataset.py`
and `python scripts/preprocess.py` first).

> Academic decision-support analysis — not an TfL operational timetable.
> Summed link loads are link-passenger exposures, **not** unique journeys.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd

from src.analysis import eda
from src.network.builder import network_statistics
from src.utils import config as cfg
from src.utils.io import load_cleaned, load_profile
from src.visualization import plots

pd.set_option("display.max_columns", 50)
profile = load_profile()
profile["cleaned_summary"] if profile else "run the profiler first"

In [ ]:
cleaned = load_cleaned()
links, line_df, station = eda.split_tables(cleaned)
links.head()

In [ ]:
stats = network_statistics(links)
print("stations:", stats["stations"], "| unique links:", stats["unique_links"])
print("links by line:", stats["links_by_line"])
print("fully connected:", stats["is_fully_connected_by_line"])
{k: v["junction_stations"] for k, v in stats["branch_analysis"].items()
 if v["junction_stations"]}

In [ ]:
sp = cfg.load_scenario_parameters()
display(eda.demand_by_time(links, line_df))
display(eda.peak_vs_offpeak(links, sp["peak_windows"]))
display(eda.frequency_by_time(links))

In [ ]:
plots.demand_heatmap(links)

In [ ]:
plots.frequency_timeseries(links)

In [ ]:
plots.demand_frequency_scatter(links)

In [ ]:
# Utilisation requires SOURCED capacities (config/capacity_by_line.csv)
cap = cfg.load_capacity_config()
util = eda.baseline_utilization(links, cap)
display(eda.utilization_summary(util))
fig = plots.utilization_heatmap(util)
fig if fig else "No capacity configured — utilisation not computed (never guessed)."

In [ ]:
# Underserved periods vs target occupancy
target = float(sp.get("target_utilization", 0.8))
eda.underserved_periods(links, cap, target).head(20)

In [ ]:
# Top demand links (peak load)
eda.demand_by_link(links, top_n=25)